# ✈️ Multi-Agent ATC Optimization — GRPO Training

**Two cooperative LLM agents** (AMAN + DMAN) learn to coordinate runway slot allocation under realistic ATC constraints.

| Component | Detail |
|-----------|--------|
| Algorithm | GRPO (Group Relative Policy Optimization) |
| Base model | Qwen2.5-7B-Instruct (4-bit QLoRA) |
| Agents | AMAN (arrivals) + DMAN (departures) + Adversarial Generator |
| Supervisor | Snorkel-style simulated expert, 5 rotating preference profiles |
| Hardware | Colab T4 (16 GB VRAM) |
| Training | ~45 min for 3 episodes × 4 tasks |

### What makes this novel
- **Single LLM** serves all 4 roles via system-prompt conditioning — no 4 separate models
- **Self-play adversarial generator** escalates difficulty when controllers improve
- **Theory-of-mind bonus**: agents rewarded for pre-emptive coordination (leaving gaps for other's emergencies)
- **Cross-lane conflict detection** on shared runways — mixed arrival/departure scheduling
- **ATFM deadline enforcement** — real network slot constraints from Eurocontrol-style flow management

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("unsloth[colab-new]", "trl>=0.9.6", "datasets>=2.20.0",
    "matplotlib>=3.9.0", "numpy>=1.26.0")
print("✅ Dependencies installed")

In [ ]:
# ── 2. Clone repo and set working directory ───────────────────────────────────
import os

REPO_URL = "https://github.com/GTsingh600/ATC"  # update to your fork

if not os.path.exists("/content/ATC"):
    subprocess.check_call(["git", "clone", REPO_URL, "/content/ATC"])

os.chdir("/content/ATC")
sys.path.insert(0, "/content/ATC")
print("✅ Repo ready:", os.getcwd())

In [ ]:
# ── 3. Verify GPU ─────────────────────────────────────────────────────────────
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    print(f"✅ GPU: {props.name} — {vram_gb:.1f} GB VRAM")
    if vram_gb < 14:
        print("⚠️  Low VRAM — reduce n_episodes or lora_rank if OOM")
else:
    print("⚠️  No GPU found — training will be very slow")

In [ ]:
# ── 4. Smoke-test the multi-agent environment ─────────────────────────────────
from multi_agent.environment import MultiAgentATCEnvironment
from multi_agent.inference import _build_aman_heuristic, _build_dman_heuristic
from tasks import task_catalog

catalog = task_catalog()
env = MultiAgentATCEnvironment(seed=42)

for task_id, task in list(catalog.items())[:2]:
    aman_obs, dman_obs = env.reset(task_id=task_id, episode_id=0)
    aman_action = _build_aman_heuristic(aman_obs)
    dman_action  = _build_dman_heuristic(dman_obs, env._state.atfm_deadlines)
    _, _, reward, _ = env.step_bid(aman_action, dman_action)
    result = env.finalize()
    print(f"{task_id}: composite={result.composite_score:.3f} "
          f"aman={result.aman_reward:.3f} dman={result.dman_reward:.3f} "
          f"coord={result.per_role.coordination_score:.3f}")

print("✅ Environment OK")

In [ ]:
# ── 5. Training configuration ─────────────────────────────────────────────────

CONFIG = dict(
    model_name      = "Qwen/Qwen2.5-7B-Instruct",
    output_dir      = "/content/outputs/atc-multiagent",
    n_episodes      = 4,        # episodes per role per training step
    lora_rank       = 16,       # 8 for tighter VRAM
    seed            = 42,
    push_to_hub     = False,    # set True + HF_TOKEN to push trained adapter
    # GRPO hypers
    num_generations = 4,        # completions per prompt for GRPO
    temperature     = 0.7,
    kl_coeff        = 0.04,
    max_new_tokens  = 512,
    learning_rate   = 2e-4,
)

import os
os.makedirs(CONFIG["output_dir"], exist_ok=True)
print("Config:", CONFIG)

In [ ]:
# ── 6. Load model with Unsloth 4-bit QLoRA ───────────────────────────────────
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = CONFIG["model_name"],
    max_seq_length = 4096,
    load_in_4bit   = True,
    dtype          = None,  # auto: bf16 on Ampere+, fp16 on T4
)

model = FastLanguageModel.get_peft_model(
    model,
    r                   = CONFIG["lora_rank"],
    target_modules      = ["q_proj", "k_proj", "v_proj", "o_proj",
                           "gate_proj", "up_proj", "down_proj"],
    lora_alpha          = CONFIG["lora_rank"] * 2,
    lora_dropout        = 0.0,
    bias                = "none",
    use_gradient_checkpointing = "unsloth",
    random_state        = CONFIG["seed"],
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"✅ Model loaded | Trainable: {trainable/1e6:.1f}M / {total/1e6:.0f}M params "
      f"({100*trainable/total:.2f}%)")

In [ ]:
# ── 7. Build multi-agent episode dataset ─────────────────────────────────────
from training.dataset import build_episode_dataset

raw_data = build_episode_dataset(
    n_episodes         = CONFIG["n_episodes"],
    seed               = CONFIG["seed"],
    include_generator  = True,
    include_supervisor = True,
)

print(f"Dataset: {len(raw_data)} prompts")
print(f"  AMAN:  {sum(1 for d in raw_data if d['role'] == 'AMAN')}")
print(f"  DMAN:  {sum(1 for d in raw_data if d['role'] == 'DMAN')}")
print(f"  GEN:   {sum(1 for d in raw_data if d['role'] == 'GENERATOR')}")
print(f"  SUP:   {sum(1 for d in raw_data if d['role'] == 'SUPERVISOR')}")

# Preview one AMAN prompt
aman_ex = next(d for d in raw_data if d["role"] == "AMAN")
print("\n--- AMAN prompt preview (first 500 chars) ---")
print(aman_ex["prompt"][:500])

In [ ]:
# ── 8. Wire reward functions ──────────────────────────────────────────────────
from training.reward_functions import (
    aman_reward_fn,
    dman_reward_fn,
    generator_reward_fn,
    supervisor_reward_fn,
)

def dispatch_reward(completions, role, **kwargs):
    """Route completions to the correct role-specific reward function."""
    fn = {
        "AMAN": aman_reward_fn,
        "DMAN": dman_reward_fn,
        "GENERATOR": generator_reward_fn,
        "SUPERVISOR": supervisor_reward_fn,
    }[role]
    return fn(completions, **kwargs)

# Quick sanity check
ex = raw_data[0]
dummy_completions = [[{"role": "assistant", "content": "invalid json"}]]
r = dispatch_reward(dummy_completions, ex["role"], **ex["reward_kwargs"])
print(f"Malformed JSON penalty: {r}  (expect ≈ -0.8)")

In [ ]:
# ── 9. GRPO training loop ─────────────────────────────────────────────────────
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset
import json
from pathlib import Path

reward_log: dict = {"AMAN": [], "DMAN": [], "GENERATOR": [], "SUPERVISOR": [], "composite": []}

def make_reward_fn(role: str, kwargs: dict):
    def _fn(completions, **_):
        scores = dispatch_reward(completions, role, **kwargs)
        reward_log[role].extend(scores)
        return scores
    return _fn

# Train each role sequentially (one LoRA adapter, role-conditioned via system prompt)
for role in ["AMAN", "DMAN", "GENERATOR"]:
    role_data = [d for d in raw_data if d["role"] == role]
    if not role_data:
        continue

    dataset = Dataset.from_list([
        {"prompt": d["messages"], **d["reward_kwargs"]} for d in role_data
    ])

    grpo_cfg = GRPOConfig(
        output_dir           = f"{CONFIG['output_dir']}/{role.lower()}",
        num_train_epochs     = 1,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        num_generations      = CONFIG["num_generations"],
        temperature          = CONFIG["temperature"],
        max_new_tokens       = CONFIG["max_new_tokens"],
        kl_coef              = CONFIG["kl_coeff"],
        learning_rate        = CONFIG["learning_rate"],
        optim                = "paged_adamw_8bit",
        bf16                 = torch.cuda.is_bf16_supported(),
        fp16                 = not torch.cuda.is_bf16_supported(),
        logging_steps        = 1,
        save_strategy        = "no",
        report_to            = "none",
        seed                 = CONFIG["seed"],
    )

    reward_fn = make_reward_fn(role, role_data[0]["reward_kwargs"])

    trainer = GRPOTrainer(
        model            = model,
        processing_class = tokenizer,
        config           = grpo_cfg,
        train_dataset    = dataset,
        reward_funcs     = [reward_fn],
    )

    print(f"\n{'='*50}\nTraining {role} ({len(dataset)} prompts)\n{'='*50}")
    trainer.train()
    mean_r = sum(reward_log[role]) / max(1, len(reward_log[role]))
    print(f"  {role} mean reward: {mean_r:.3f}")

# Save adapter
model.save_pretrained(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])

# Save reward curves
curves_path = Path(CONFIG["output_dir"]) / "reward_curves.json"
curves_path.write_text(json.dumps(reward_log, indent=2))
print(f"\n✅ Training complete | Adapter saved to {CONFIG['output_dir']}")
print(f"   Reward curves: {curves_path}")

In [ ]:
# ── 10. Plot reward curves ─────────────────────────────────────────────────────
from training.plot_rewards import plot_training_curves
import json
from pathlib import Path

curves = json.loads(curves_path.read_text())
plot_training_curves(curves, save_dir=CONFIG["output_dir"], show=True)

In [ ]:
# ── 11. Before / after evaluation ─────────────────────────────────────────────
from training.eval import evaluate_model, print_comparison

EVAL_TASKS = ["delhi_monsoon_recovery_easy", "bengaluru_irrops_hard"]
N_EVAL_EPS = 3

print("Running base (heuristic) evaluation...")
base_results = evaluate_model(
    "heuristic-baseline", N_EVAL_EPS, EVAL_TASKS, seed=99,
    use_generator=True, label="BASE",
)

print("\nRunning trained model evaluation...")
FastLanguageModel.for_inference(model)
trained_results = evaluate_model(
    CONFIG["output_dir"], N_EVAL_EPS, EVAL_TASKS, seed=99,
    use_generator=True, label="TRAINED",
)

print_comparison(base_results, trained_results)

In [ ]:
# ── 12. Before/after bar chart ─────────────────────────────────────────────────
from training.plot_rewards import plot_eval_comparison
import json
from pathlib import Path

eval_out = {
    "base":    {k: v for k, v in base_results.items()    if k != "records"},
    "trained": {k: v for k, v in trained_results.items() if k != "records"},
}
eval_path = Path(CONFIG["output_dir"]) / "eval_results.json"
eval_path.write_text(json.dumps(eval_out, indent=2))

plot_eval_comparison(eval_out, save_dir=CONFIG["output_dir"], show=True)
print(f"Eval results saved to {eval_path}")

In [ ]:
# ── 13. (Optional) Push adapter to Hugging Face Hub ──────────────────────────
if CONFIG["push_to_hub"]:
    HF_TOKEN = "hf_..."   # paste your token or use userdata.get('HF_TOKEN')
    HUB_REPO = "your-username/atc-multiagent-grpo"

    model.push_to_hub(HUB_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HUB_REPO, token=HF_TOKEN)
    print(f"✅ Pushed to https://huggingface.co/{HUB_REPO}")
else:
    print("Skipping hub push (set CONFIG['push_to_hub'] = True to enable)")

## Expected results

After ~45 min of training on T4:

| Metric | Base (heuristic) | Trained (GRPO) | Expected delta |
|--------|-----------------|----------------|----------------|
| Composite score | ~0.52 | ~0.68 | ↑ +0.16 |
| AMAN reward | ~0.55 | ~0.70 | ↑ +0.15 |
| DMAN reward | ~0.50 | ~0.65 | ↑ +0.15 |
| Coordination score | ~0.40 | ~0.62 | ↑ +0.22 |
| Success rate (≥0.60) | ~35% | ~68% | ↑ +33pp |

### What GRPO learns
- **AMAN** learns to leave runway gaps at emergency-probable times (theory-of-mind)
- **DMAN** learns to broadcast ATFM constraints proactively rather than waiting for conflicts
- **Generator** escalates difficulty as controllers improve — self-play arms race
- **Supervisor** preference profile rotation prevents reward hacking on a single metric

### Key files
- [`multi_agent/environment.py`](multi_agent/environment.py) — core AMAN/DMAN env
- [`multi_agent/models.py`](multi_agent/models.py) — all data models
- [`training/reward_functions.py`](training/reward_functions.py) — GRPO reward fns
- [`training/dataset.py`](training/dataset.py) — episode dataset builder
- [`tests/test_multi_agent.py`](tests/test_multi_agent.py) — 20+ smoke tests